In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

from sklearn.metrics import r2_score,root_mean_squared_error, mean_squared_error, mean_absolute_error
from sklearn.model_selection import LeaveOneGroupOut

In [17]:
#File checker
DATA_DIR = Path("data")
data_files = sorted(
    p for p in DATA_DIR.iterdir()
    if p.suffix.lower() in {".csv"}#, ".parquet", ".pkl", ".pickle"}
)
df_raw = pd.read_csv(
    DATA_DIR / "perera_pfizer_raw_download.csv"
)
# df_raw = pd.read_csv(
#     PERERA_URL,
#     sep=";",
#     decimal=",",
#     encoding="utf-8-sig"
# )
print("Directory found:", DATA_DIR)
print("scikitlearn version:", sklearn.__version__)
print("Files found:")
for p in data_files:
    print(p.name)
    print(pd.read_csv(p, sep=",", decimal=",", encoding="utf-8-sig").shape)
    print(pd.read_csv(p, sep=",", decimal=",", encoding="utf-8-sig").head(1).columns.tolist())

Directory found: data
scikitlearn version: 1.9.0
Files found:
kraken_features_only.csv
(1223, 191)
['id', 'vmin_vmin_boltz', 'vmin_r_boltz', 'fmo_e_homo_boltz', 'fmo_e_lumo_boltz', 'fmo_mu_boltz', 'fmo_eta_boltz', 'fmo_omega_boltz', 'somo_ra_boltz', 'somo_rc_boltz', 'nbo_P_boltz', 'nbo_P_ra_boltz', 'spindens_P_ra_boltz', 'nbo_P_rc_boltz', 'spindens_P_rc_boltz', 'nmr_P_boltz', 'nmrtens_sxx_P_boltz', 'nmrtens_syy_P_boltz', 'nmrtens_szz_P_boltz', 'efg_amp_P_boltz', 'efgtens_xx_P_boltz', 'efgtens_yy_P_boltz', 'efgtens_zz_P_boltz', 'nuesp_P_boltz', 'E_solv_cds_boltz', 'nbo_lp_P_percent_s_boltz', 'nbo_lp_P_occ_boltz', 'nbo_lp_P_e_boltz', 'nbo_bd_e_max_boltz', 'nbo_bd_e_avg_boltz', 'nbo_bds_e_min_boltz', 'nbo_bds_e_avg_boltz', 'nbo_bd_occ_min_boltz', 'nbo_bd_occ_avg_boltz', 'nbo_bds_occ_max_boltz', 'nbo_bds_occ_avg_boltz', 'E_solv_total_boltz', 'E_solv_elstat_boltz', 'E_oxidation_boltz', 'E_reduction_boltz', 'fukui_p_boltz', 'fukui_m_boltz', 'vbur_vtot_boltz', 'vbur_ratio_vbur_vtot_boltz', 'P

In [18]:
#Load in data sets
df_interp= pd.read_csv(DATA_DIR /"perera_pfizer_group1_kraken_interpretable.csv")
df_full = pd.read_csv(DATA_DIR / "perera_pfizer_group1_kraken_full.csv")

#drop static  features
df_interp = df_interp.loc[:, df_interp.nunique(dropna=False) > 1].copy()
df_full = df_full.loc[:, df_full.nunique(dropna=False) > 1].copy()
 
#Target variable
TARGET = "Product_Yield_PCT_Area_UV"

#Non catalyst variables
non_catalyst = ["Reactant_2_Name", "Reactant_1_Name", "Solvent_1_Short_Hand", "ligand"]

#Drop additional yield feature
df_interp = df_interp.drop(columns=["Product_Yield_Mass_Ion_Count"])
df_full = df_full.drop(columns=["Product_Yield_Mass_Ion_Count"])

#Drop non needed reaction based features
df_interp = df_interp.drop(columns=["Reaction_No","Reagent_combo","reactant1_code","reactant2_code","Reactant_1_Name","Reactant_2_Name"])#,"kraken_id","Reactant_1_Short_Hand","Reagent_1_Short_Hand",,"substrate_pair",])
df_full = df_full.drop(columns=["Reaction_No","Reactant_1_Short_Hand","Reagent_combo","reactant1_code","reactant2_code","Reactant_1_Name","Reactant_2_Name"])#"kraken_id","Reagent_1_Short_Hand",,"substrate_pair",])
#Drop to make one hot encoding
df_onehot_ligand = df_interp.copy().drop(columns=["vmin_vmin_boltz","fmo_e_homo_boltz","fmo_e_lumo_boltz","%vbur_max","dipolemoment_boltz","%vbur_boltz","%vbur_min","%vbur_delta","delta_band_gap"])#,"kraken_id"
# for PCA connection
df_onehot_id = df_interp.copy().drop(columns=["kraken_id","dipolemoment_boltz","%vbur_boltz","%vbur_min","%vbur_delta","delta_band_gap","vmin_vmin_boltz","fmo_e_homo_boltz","fmo_e_lumo_boltz","%vbur_max"])#,""
#Drop catalyst features in interpretable set to only the 5 permitted
#(keep "ligand" as an identity column for LOLO grouping -- build_Xy excludes it
# from the feature matrix for every model except df_onehot_ligand)
df_interp = df_interp.drop(columns=["vmin_vmin_boltz","fmo_e_homo_boltz","fmo_e_lumo_boltz","%vbur_max"])
#Drop all but three 
df_bur_boltz_vbur_min_vbur_delta = df_interp.copy().drop(columns=["delta_band_gap","dipolemoment_boltz"])
# Drop all but two 
df_bur_boltz_vbur_min = df_interp.copy().drop(columns=["delta_band_gap","dipolemoment_boltz","%vbur_delta"])


In [19]:
for feature in df_interp.columns:
    print(f"{feature}: {df_interp[feature].nunique(dropna=False)} unique entries")

ligand: 8 unique entries
Reactant_1_Short_Hand: 4 unique entries
Reagent_1_Short_Hand: 8 unique entries
Solvent_1_Short_Hand: 4 unique entries
Product_Yield_PCT_Area_UV: 2510 unique entries
substrate_pair: 12 unique entries
kraken_id: 8 unique entries
dipolemoment_boltz: 8 unique entries
%vbur_boltz: 8 unique entries
%vbur_min: 8 unique entries
%vbur_delta: 6 unique entries
delta_band_gap: 8 unique entries


In [20]:
for col in df_interp.columns:
    print(f"\n--- {col} ---")
    print(df_interp[col].value_counts(dropna=False))


--- ligand ---
ligand
P(tBu)3        384
P(Ph)3         384
AmPhos         384
P(Cy)3         384
P(o-Tol)3      384
CataCXium A    384
SPhos          384
XPhos          384
Name: count, dtype: int64

--- Reactant_1_Short_Hand ---
Reactant_1_Short_Hand
1a, 6-Cl-Q     768
1b, 6-Br-Q     768
1c, 6-OTf-Q    768
1d, 6-I-Q      768
Name: count, dtype: int64

--- Reagent_1_Short_Hand ---
Reagent_1_Short_Hand
NaOH      384
NaHCO3    384
CsF       384
K3PO4     384
KOH       384
LiOtBu    384
Et3N      384
NaN       384
Name: count, dtype: int64

--- Solvent_1_Short_Hand ---
Solvent_1_Short_Hand
MeCN    768
THF     768
DMF     768
MeOH    768
Name: count, dtype: int64

--- Product_Yield_PCT_Area_UV ---
Product_Yield_PCT_Area_UV
70.45    5
13.04    5
15.05    5
17.77    5
13.53    5
        ..
18.14    1
17.83    1
20.89    1
48.66    1
43.45    1
Name: count, Length: 2510, dtype: int64

--- substrate_pair ---
substrate_pair
1a_2a    256
1b_2a    256
1c_2a    256
1d_2a    256
1c_2b    256
1d_2

In [21]:
for feature in df_interp.columns:
    print(f"{feature}: {df_interp[feature].nunique(dropna=False)} unique entries")

ligand: 8 unique entries
Reactant_1_Short_Hand: 4 unique entries
Reagent_1_Short_Hand: 8 unique entries
Solvent_1_Short_Hand: 4 unique entries
Product_Yield_PCT_Area_UV: 2510 unique entries
substrate_pair: 12 unique entries
kraken_id: 8 unique entries
dipolemoment_boltz: 8 unique entries
%vbur_boltz: 8 unique entries
%vbur_min: 8 unique entries
%vbur_delta: 6 unique entries
delta_band_gap: 8 unique entries


In [22]:
for feature in df_full.columns:
    print(f"{feature}: {df_full[feature].nunique(dropna=False)} unique entries")

Ligand_Short_Hand: 8 unique entries
Reagent_1_Short_Hand: 8 unique entries
Solvent_1_Short_Hand: 4 unique entries
Product_Yield_PCT_Area_UV: 2510 unique entries
substrate_pair: 12 unique entries
kraken_id: 8 unique entries
vmin_vmin_boltz: 8 unique entries
vmin_r_boltz: 8 unique entries
fmo_e_homo_boltz: 8 unique entries
fmo_e_lumo_boltz: 8 unique entries
fmo_mu_boltz: 8 unique entries
fmo_eta_boltz: 8 unique entries
fmo_omega_boltz: 8 unique entries
somo_ra_boltz: 8 unique entries
somo_rc_boltz: 8 unique entries
nbo_P_boltz: 8 unique entries
nbo_P_ra_boltz: 8 unique entries
spindens_P_ra_boltz: 8 unique entries
nbo_P_rc_boltz: 8 unique entries
spindens_P_rc_boltz: 8 unique entries
nmr_P_boltz: 8 unique entries
nmrtens_sxx_P_boltz: 8 unique entries
nmrtens_syy_P_boltz: 8 unique entries
nmrtens_szz_P_boltz: 8 unique entries
efg_amp_P_boltz: 8 unique entries
efgtens_xx_P_boltz: 8 unique entries
efgtens_yy_P_boltz: 8 unique entries
efgtens_zz_P_boltz: 8 unique entries
nuesp_P_boltz: 8 uni

In [23]:
# ---- Fit PCA on the full kraken descriptor table and expose the loadings ----
# kraken_features_only.csv = one row per ligand ("id") + the 190 physical-organic descriptors.
# Per the SI, descriptors are standard-scaled BEFORE PCA.

df_desc = pd.read_csv(DATA_DIR / "kraken_features_only.csv")

# Descriptor columns = everything except the ligand identifier
DESC_ID_COL = "id"
desc_cols = [c for c in df_desc.columns if c != DESC_ID_COL]

# Drop any descriptor that is entirely NaN, then fill remaining gaps with column means
X_desc = df_desc[desc_cols].apply(pd.to_numeric, errors="coerce")
X_desc = X_desc.dropna(axis=1, how="all")
desc_cols = X_desc.columns.tolist()
X_desc = X_desc.fillna(X_desc.mean())

N_COMPONENTS = 10  # keep the first 10 PCs (paper reports variance for PC1..PC10)

pca_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS, random_state=42)),
])
scores = pca_pipe.fit_transform(X_desc)                 # per-ligand PC scores
pca = pca_pipe.named_steps["pca"]

pc_names = [f"PC{i+1}" for i in range(N_COMPONENTS)]

# Loadings: rows = original descriptors, cols = PCs. (components_ is [n_pc, n_features])
loadings = pd.DataFrame(pca.components_.T, index=desc_cols, columns=pc_names)

# Explained variance (sanity check against Table S7: ~28.0, 12.9, 11.2, 6.6 ...)
explained = pd.Series(pca.explained_variance_ratio_, index=pc_names, name="explained_var_ratio")
print("Explained variance ratio:")
print((explained * 100).round(1).to_string())
loadings.head()


Explained variance ratio:
PC1     29.0
PC2     13.0
PC3     11.2
PC4      6.1
PC5      4.8
PC6      4.5
PC7      3.5
PC8      2.5
PC9      2.3
PC10     1.9


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
vmin_vmin_boltz,-0.040673,0.109820,-0.041338,0.131300,0.016202,0.037612,0.070836,-0.161692,-0.050804,0.021805
vmin_r_boltz,-0.031977,0.094786,-0.029553,0.106694,0.001846,-0.010934,0.078246,-0.204389,-0.051469,0.006858
fmo_e_homo_boltz,0.068985,-0.080719,0.007620,-0.067753,-0.037611,0.016744,-0.103893,0.172853,0.105747,0.105483
fmo_e_lumo_boltz,-0.051804,-0.065353,0.107962,-0.040589,-0.043974,0.065513,-0.012614,0.166571,0.084007,-0.103619
fmo_mu_boltz,0.000139,-0.092731,0.083557,-0.067352,-0.053145,0.057416,-0.066810,0.218140,0.120301,-0.017723


In [24]:
# ---- Rank feature contributions for any PC ----
def rank_loadings(pc, n=None, by_abs=True):
    """Return descriptors ranked by their loading on `pc` (e.g. 'PC1')."""
    s = loadings[pc]
    order = s.abs().sort_values(ascending=False) if by_abs else s.sort_values(ascending=False)
    out = pd.DataFrame({"loading": s.loc[order.index], "abs_loading": s.loc[order.index].abs()})
    return out.head(n) if n else out

# Example: top 10 contributors to PC1
rank_loadings("PC1", 10)


,loading,abs_loading
vbur_vtot_boltz,0.124170,0.124170
volume_boltz,0.123924,0.123924
vbur_near_vtot_vburminconf,0.122562,0.122562
vbur_near_vtot_max,0.122505,0.122505
surface_area_boltz,0.122362,0.122362
Pint_P_max_boltz,0.119212,0.119212
Pint_P_int_boltz,0.117934,0.117934
vbur_qvtot_max_max,0.116291,0.116291
vbur_qvtot_max_boltz,0.115987,0.115987
vbur_qvtot_max_vburminconf,0.115516,0.115516


In [25]:
# ---- Top-3 descriptors from each of PC1..PC4 -> merge onto the interpretable / one-hot table ----
TOP_K = 3
TOP_PCS = ["PC1", "PC2", "PC3", "PC4"]

top_desc_per_pc = {pc: rank_loadings(pc, TOP_K).index.tolist() for pc in TOP_PCS}
for pc, cols in top_desc_per_pc.items():
    print(pc, "->", cols)

# De-duplicated, order-preserving list of the raw descriptor columns to add
pc_top_features = list(dict.fromkeys(c for cols in top_desc_per_pc.values() for c in cols))
print("\nDescriptors added from PCs:", pc_top_features)

# Per-ligand lookup table of just those descriptors, keyed by kraken id
desc_lookup = df_desc[[DESC_ID_COL] + pc_top_features].copy()

# Choose the base table to augment. Prefer df_onehot_ligand if you've already built it;
# otherwise fall back to df_interp. Both are keyed to a ligand via kraken id.
base_df = df_onehot_ligand.copy() if "df_onehot_ligand" in dir() else df_interp.copy()

# Figure out which column in base_df holds the kraken ligand id.
id_candidates = [c for c in ("kraken_id", "id", "ligand") if c in base_df.columns]
if not id_candidates:
    raise KeyError("No kraken id column ('kraken_id'/'id'/'ligand') found in base_df to merge on.")
BASE_ID_COL = id_candidates[0]

df_pca_augmented = base_df.merge(
    desc_lookup, how="left", left_on=BASE_ID_COL, right_on=DESC_ID_COL,
)
if DESC_ID_COL != BASE_ID_COL and DESC_ID_COL in df_pca_augmented.columns:
    df_pca_augmented = df_pca_augmented.drop(columns=[DESC_ID_COL])

missing = df_pca_augmented[pc_top_features].isna().any(axis=1).sum()
print(f"\nMerged on '{BASE_ID_COL}'. Rows with unmatched descriptors: {missing}")
print("df_pca_augmented shape:", df_pca_augmented.shape)
df_pca_augmented.head()


PC1 -> ['vbur_vtot_boltz', 'volume_boltz', 'vbur_near_vtot_vburminconf']
PC2 -> ['pyr_P_max', 'qpole_amp_max', 'vbur_qvbur_min_min']
PC3 -> ['vbur_near_vbur_delta', 'vbur_vbur_delta', 'vbur_qvbur_min_delta']
PC4 -> ['nbo_bds_occ_avg_boltz', 'efgtens_zz_P_boltz', 'nmr_P_boltz']

Descriptors added from PCs: ['vbur_vtot_boltz', 'volume_boltz', 'vbur_near_vtot_vburminconf', 'pyr_P_max', 'qpole_amp_max', 'vbur_qvbur_min_min', 'vbur_near_vbur_delta', 'vbur_vbur_delta', 'vbur_qvbur_min_delta', 'nbo_bds_occ_avg_boltz', 'efgtens_zz_P_boltz', 'nmr_P_boltz']

Merged on 'kraken_id'. Rows with unmatched descriptors: 0
df_pca_augmented shape: (3072, 19)


,ligand,Reactant_1_Short_Hand,Reagent_1_Short_Hand,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,substrate_pair,kraken_id,vbur_vtot_boltz,volume_boltz,vbur_near_vtot_vburminconf,pyr_P_max,qpole_amp_max,vbur_qvbur_min_min,vbur_near_vbur_delta,vbur_vbur_delta,vbur_qvbur_min_delta,nbo_bds_occ_avg_boltz,efgtens_zz_P_boltz,nmr_P_boltz
0,P(tBu)3,"1a, 6-Cl-Q",NaOH,MeCN,4.76,1a_2a,8,253.200425,317.844460,253.200425,0.830704,2.814976,15.199086,0.000000,0.000000,0.000000,0.052453,1.428239,225.189000
1,P(Ph)3,"1a, 6-Cl-Q",NaOH,MeCN,4.12,1a_2a,17,302.715907,344.207520,302.715907,0.932082,4.561356,11.489263,0.000000,0.000000,0.000000,0.037103,1.456498,296.152700
2,AmPhos,"1a, 6-Cl-Q",NaOH,MeCN,2.58,1a_2a,216,318.405750,390.247930,316.760137,0.855551,12.311883,13.548649,0.000000,0.000000,0.000000,0.050737,1.432126,256.655600
3,P(Cy)3,"1a, 6-Cl-Q",NaOH,MeCN,4.44,1a_2a,11,325.641196,412.898679,325.712062,0.937360,4.044921,9.958058,12.226625,17.283574,3.696227,0.032226,1.372408,283.164838
4,P(o-Tol)3,"1a, 6-Cl-Q",NaOH,MeCN,1.95,1a_2a,9,352.641630,407.229542,349.914566,0.918251,6.600232,11.333423,8.174791,10.024996,4.617669,0.041685,1.456571,317.294407


## GP yield models — shared setup + three scoring schemes

Structure of each scoring cell is the same:

**set hyperparameters → loop over each dataframe → calculate the R² for each model**

The kernel and the dataframe registry are defined once here, so you can change the kernel
in one place (`make_kernel`) or add another dataframe to `DATAFRAMES` and every scoring
cell below picks it up with no other edits.


In [26]:
# ===== SHARED GP SETUP (edit hyperparameters here) =====================================
from sklearn.model_selection import KFold
from sklearn.base import clone

TARGET = "Product_Yield_PCT_Area_UV"

# --- Kernel: change this one function to swap kernels everywhere ---------------------
def make_kernel(n_features):
    """Return a fresh GP kernel. Swap RBF for Matern/ConstantKernel*RBF etc. here later."""
    return RBF(length_scale=np.ones(n_features), length_scale_bounds=(1e-2, 1e3))
    # examples for later:
    # return ConstantKernel(1.0) * RBF(np.ones(n_features)) + WhiteKernel()
    # return Matern(length_scale=np.ones(n_features), nu=2.5) + WhiteKernel()

# --- GP hyperparameters -------------------------------------------------------------
GP_KWARGS = dict(
    alpha=1e-6,                 # jitter added to diagonal (raise if Cholesky fails)
    n_restarts_optimizer=0,     # kernel hyperparameter restarts
    normalize_y=True,           # standardize y internally
    random_state=42,
)

# --- Registry of dataframes to model. Append here to add more later. ----------------
DATAFRAMES = {
    "df_onehot_ligand": df_onehot_ligand,
    "df_bur_boltz_vbur_min": df_bur_boltz_vbur_min,
    "df_bur_boltz_vbur_min_vbur_delta": df_bur_boltz_vbur_min_vbur_delta,
    "df_interp":        df_interp,
    #"df_full":          df_full,
    #"df_pca_augmented": df_pca_augmented,
}

# --- Column used to group folds for leave-one-ligand-out ----------------------------
LIGAND_COL = "ligand"

# Only these dataframes are allowed to use ligand identity as a training feature
# (df_onehot_ligand one-hot-encodes it). Every other dataframe still carries the
# "ligand" column so LOLO can group by it, but build_Xy drops it from the feature
# matrix for them.
LIGAND_FEATURE_MODELS = {"df_onehot_ligand"}

# Set to a specific ligand name (e.g. "SPhos") to run LOLO holding out only that
# ligand; leave as None to run the full leave-one-ligand-out loop over every ligand.
HOLDOUT_LIGAND = "AmPhos"

# --- Build (X, y, groups) from a dataframe: one-hot categoricals, scale numerics -----
def build_Xy(df, name=None):
    """Return (preprocessor, X_frame, y, groups) for a given dataframe.

    `name` is the DATAFRAMES registry key -- used only to decide whether the
    "ligand" column is allowed to stay in the feature matrix (see LIGAND_FEATURE_MODELS).
    """
    d = df.copy()
    y = d[TARGET].to_numpy(dtype=float)

    # group labels for leave-one-ligand-out (fall back to None if absent)
    groups = d[LIGAND_COL].to_numpy() if LIGAND_COL in d.columns else None

    # feature frame = everything except target and obvious id/leakage columns
    drop_cols = [c for c in [TARGET, "kraken_id", "id"] if c in d.columns]
    if LIGAND_COL in d.columns and name not in LIGAND_FEATURE_MODELS:
        drop_cols.append(LIGAND_COL)
    X = d.drop(columns=drop_cols)

    # split numeric vs categorical (avoid select_dtypes(include=["object"]) which
    # triggers the pandas object/str deprecation warning -- everything non-numeric
    # is treated as categorical instead)
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    pre = ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])
    return pre, X, y, groups

def make_gp(n_features):
    return GaussianProcessRegressor(kernel=make_kernel(n_features), **GP_KWARGS)

print("Setup ready. Dataframes registered:", list(DATAFRAMES))

# --- Print each dataframe, its feature count, and first N feature names once, before any model is trained ---
N_PREVIEW_FEATURES = 10
for name, df in DATAFRAMES.items():
    _, X, _, _ = build_Xy(df, name)
    print(f"\n=== {name} ===")
    #print(df)
    print(f"{name}: {len(X.columns)} features used to train -> first {N_PREVIEW_FEATURES}: {X.columns[:N_PREVIEW_FEATURES].tolist()}")


Setup ready. Dataframes registered: ['df_onehot_ligand', 'df_bur_boltz_vbur_min', 'df_bur_boltz_vbur_min_vbur_delta', 'df_interp']

=== df_onehot_ligand ===
df_onehot_ligand: 5 features used to train -> first 10: ['ligand', 'Reactant_1_Short_Hand', 'Reagent_1_Short_Hand', 'Solvent_1_Short_Hand', 'substrate_pair']

=== df_bur_boltz_vbur_min ===
df_bur_boltz_vbur_min: 6 features used to train -> first 10: ['Reactant_1_Short_Hand', 'Reagent_1_Short_Hand', 'Solvent_1_Short_Hand', 'substrate_pair', '%vbur_boltz', '%vbur_min']

=== df_bur_boltz_vbur_min_vbur_delta ===
df_bur_boltz_vbur_min_vbur_delta: 7 features used to train -> first 10: ['Reactant_1_Short_Hand', 'Reagent_1_Short_Hand', 'Solvent_1_Short_Hand', 'substrate_pair', '%vbur_boltz', '%vbur_min', '%vbur_delta']

=== df_interp ===
df_interp: 9 features used to train -> first 10: ['Reactant_1_Short_Hand', 'Reagent_1_Short_Hand', 'Solvent_1_Short_Hand', 'substrate_pair', 'dipolemoment_boltz', '%vbur_boltz', '%vbur_min', '%vbur_delta',

In [16]:
# ===== SCORING 1: LEAVE-ONE-LIGAND-OUT =================================================
# set hyperparameters (from shared setup) -> loop over each dataframe -> R^2 per model
# HOLDOUT_LIGAND (set in the shared setup cell) picks a single ligand to hold out;
# leave it as None to run the full leave-one-ligand-out loop over every ligand.
logo = LeaveOneGroupOut()
results_loo = {}

for name, df in DATAFRAMES.items():
    pre, X, y, groups = build_Xy(df, name)
    if groups is None:
        print(f"{name}: no '{LIGAND_COL}' column -> skipping leave-one-ligand-out")
        continue

    folds = list(logo.split(X, y, groups))
    if HOLDOUT_LIGAND is not None:
        if HOLDOUT_LIGAND not in groups:
            print(f"{name}: ligand '{HOLDOUT_LIGAND}' not found -> skipping")
            continue
        folds = [(tr, te) for tr, te in folds if set(groups[te]) == {HOLDOUT_LIGAND}]

    y_true_all, y_pred_all = [], []
    for tr, te in folds:
        Xtr = pre.fit_transform(X.iloc[tr]); Xte = pre.transform(X.iloc[te])
        gp = make_gp(Xtr.shape[1]).fit(Xtr, y[tr])
        y_pred_all.append(gp.predict(Xte)); y_true_all.append(y[te])

    r2 = r2_score(np.concatenate(y_true_all), np.concatenate(y_pred_all))
    results_loo[name] = r2
    held = f"'{HOLDOUT_LIGAND}'" if HOLDOUT_LIGAND is not None else f"all {len(np.unique(groups))} ligands"
    print(f"{name}: leave-one-ligand-out R^2 = {r2:.3f}  (held out: {held})")

print("\nSummary (LOLO):", {k: round(v, 3) for k, v in results_loo.items()})


KeyboardInterrupt: 

In [ ]:
# ===== SCORING 2: 5-FOLD CROSS-VALIDATION =============================================
# set hyperparameters -> loop over each dataframe -> R^2 per model
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results_kfold = {}

for name, df in DATAFRAMES.items():
    pre, X, y, _ = build_Xy(df, name)

    y_true_all, y_pred_all = [], []
    for tr, te in kf.split(X):
        Xtr = pre.fit_transform(X.iloc[tr]); Xte = pre.transform(X.iloc[te])
        gp = make_gp(Xtr.shape[1]).fit(Xtr, y[tr])
        y_pred_all.append(gp.predict(Xte)); y_true_all.append(y[te])

    r2 = r2_score(np.concatenate(y_true_all), np.concatenate(y_pred_all))
    results_kfold[name] = r2
    print(f"{name}: 5-fold R^2 = {r2:.3f}")

print("\nSummary (5-fold):", {k: round(v, 3) for k, v in results_kfold.items()})


In [ ]:
# ===== SCORING 3: FIT-AND-SCORE (in-sample sanity check) ==============================
# set hyperparameters -> loop over each dataframe -> R^2 per model
# NOTE: trains and scores on the SAME data -> optimistic; use only as a sanity check.
results_insample = {}

for name, df in DATAFRAMES.items():
    pre, X, y, _ = build_Xy(df, name)
    Xt = pre.fit_transform(X)
    gp = make_gp(Xt.shape[1]).fit(Xt, y)
    r2 = r2_score(y, gp.predict(Xt))
    results_insample[name] = r2
    print(f"{name}: in-sample R^2 = {r2:.3f}")

print("\nSummary (in-sample):", {k: round(v, 3) for k, v in results_insample.items()})


In [ ]:
# ===== SCORING 4: 80:20 TRAIN-TEST SPLIT ================================================
# set hyperparameters -> loop over each dataframe -> R^2 per model
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.2
results_traintest = {}

for name, df in DATAFRAMES.items():
    pre, X, y, _ = build_Xy(df, name)
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=42)

    Xtr = pre.fit_transform(Xtr); Xte = pre.transform(Xte)
    gp = make_gp(Xtr.shape[1]).fit(Xtr, ytr)
    r2 = r2_score(yte, gp.predict(Xte))
    results_traintest[name] = r2
    print(f"{name}: 80:20 train-test R^2 = {r2:.3f}")

print("\nSummary (80:20 train-test):", {k: round(v, 3) for k, v in results_traintest.items()})
